# Evaluate the Fine-tuned OPT Model

This notebook is independent of the training kernel. It loads the saved tokenizer and LoRA adapter, generates one structured response per test article, evaluates the summary and six aspects, and saves per-example and aggregate results.

## Setup

In [1]:
# %pip install -r requirements.txt

In [2]:
MODEL_NAME = "opt-125m-cnn-news-simplifier"
RESULT_FILE_NAME = "opt125m_cnn_test_results"                                                  

In [3]:
import json
import os
import re
import time
from datetime import datetime
from pathlib import Path

import pandas as pd
import textstat
import torch
from peft import PeftConfig, PeftModel
from rouge_score import rouge_scorer
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)

TEST_PATH = Path("data/processed/test.jsonl")
TEMPLATES_PATH = Path("data/processed/prompt_templates.json")
FINETUNED_MODEL_DIR = Path(f"models/{MODEL_NAME}")
RESULTS_DIR = Path("results")

MAX_SEQUENCE_LENGTH = 2048
MAX_NEW_TOKENS = 384

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

W0802 15:51:25.583000 121476 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


PyTorch: 2.13.0+cu126
CUDA available: True
GPU: NVIDIA GeForce RTX 4060 Laptop GPU


## Load the test split, prompt template, and saved tokenizer

In [4]:
def require_path(path: Path, description: str) -> None:
    if not path.exists():
        raise FileNotFoundError(f"Missing {description}: '{path}'")


require_path(TEST_PATH, "test split")
require_path(TEMPLATES_PATH, "prompt template")
require_path(FINETUNED_MODEL_DIR / "adapter_config.json", "adapter configuration")
require_path(FINETUNED_MODEL_DIR / "adapter_model.safetensors", "adapter weights")

records = []
with TEST_PATH.open("r", encoding="utf-8") as file:
    for line_number, line in enumerate(file, start=1):
        if line.strip():
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError as error:
                raise ValueError(
                    f"Invalid JSON in {TEST_PATH} at line {line_number}."
                ) from error

test_df = pd.DataFrame(records)
required_columns = {
    "article_id",
    "article",
    "summary",
    "article_word_count",
    "aspects",
}
missing_columns = required_columns - set(test_df.columns)
if missing_columns:
    raise ValueError(f"Test split is missing columns: {sorted(missing_columns)}")
if test_df.empty:
    raise ValueError("The test split is empty.")
if test_df["article_id"].duplicated().any():
    raise ValueError("The test split contains duplicate article IDs.")
if not test_df["aspects"].map(lambda value: isinstance(value, dict)).all():
    raise TypeError("Every test record must contain an 'aspects' dictionary.")

with TEMPLATES_PATH.open("r", encoding="utf-8") as file:
    prompt_templates = json.load(file)

if "prompt_template" not in prompt_templates:
    raise KeyError("prompt_templates.json must contain 'prompt_template'.")
PROMPT_TEMPLATE = prompt_templates["prompt_template"]
if PROMPT_TEMPLATE.count("{article}") != 1:
    raise ValueError(
        "The prompt template must contain exactly one '{article}' placeholder."
    )
PROMPT_PREFIX, PROMPT_SUFFIX = PROMPT_TEMPLATE.split("{article}")

tokenizer = AutoTokenizer.from_pretrained(
    FINETUNED_MODEL_DIR,
    use_fast=True,
)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

SUMMARY_GENERATION_PREFIX = "Summary:\n"
prefix_ids = tokenizer(PROMPT_PREFIX, add_special_tokens=True)["input_ids"]
generation_suffix_ids = tokenizer(
    PROMPT_SUFFIX + SUMMARY_GENERATION_PREFIX,
    add_special_tokens=False,
)["input_ids"]

ASPECT_LABELS = {
    "event": "Event",
    "people": "People",
    "date_time": "Date & Time",
    "location": "Location",
    "effects": "Effects",
    "countermeasures": "Countermeasures",
}

print(f"Loaded {len(test_df)} test records.")
print(f"Tokenizer: {tokenizer.name_or_path}")
print(f"Prompt template: {TEMPLATES_PATH}")

Loaded 30 test records.
Tokenizer: models\opt-125m-cnn-news-simplifier
Prompt template: data\processed\prompt_templates.json


## Load the saved base model and LoRA adapter

The adapter configuration records the exact base checkpoint used during training. On a CUDA machine the base model is loaded in the same 4-bit NF4 format used for QLoRA; CPU loading is supported as a slower fallback.

In [5]:
peft_config = PeftConfig.from_pretrained(FINETUNED_MODEL_DIR)
base_model_name = peft_config.base_model_name_or_path
if not base_model_name:
    raise ValueError("The adapter configuration does not identify a base model.")

if torch.cuda.is_available():
    compute_dtype = (
        torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    )
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=compute_dtype,
    )
    base_model = AutoModelForCausalLM.from_pretrained(
        base_model_name,
        quantization_config=quantization_config,
        device_map="auto",
    )
else:
    print("CUDA is unavailable; evaluation will run slowly on CPU.")
    base_model = AutoModelForCausalLM.from_pretrained(base_model_name)

finetuned_model = PeftModel.from_pretrained(
    base_model,
    FINETUNED_MODEL_DIR,
    is_trainable=False,
)
finetuned_model.eval()
finetuned_model.config.use_cache = True
model_device = finetuned_model.get_input_embeddings().weight.device
context_limit = finetuned_model.config.max_position_embeddings
if MAX_SEQUENCE_LENGTH > context_limit:
    raise ValueError(
        f"MAX_SEQUENCE_LENGTH={MAX_SEQUENCE_LENGTH} exceeds the model "
        f"context limit of {context_limit}."
    )
if MAX_NEW_TOKENS >= MAX_SEQUENCE_LENGTH:
    raise ValueError("MAX_NEW_TOKENS must be smaller than MAX_SEQUENCE_LENGTH.")

print(f"Base model: {base_model_name}")
print(f"Adapter: {FINETUNED_MODEL_DIR}")
print(f"Model input device: {model_device}")

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

Base model: facebook/opt-125m
Adapter: models\opt-125m-cnn-news-simplifier
Model input device: cuda:0


## Generate summaries and aspects for the test split

Each article is sent to the model once. Generation is prefixed with `Summary:` so the model begins in the required section. The complete response is retained unchanged in `generated_text`; only the summary boundary is parsed, while the remaining aspect block is retained unchanged in `generated_aspects`.

In [6]:
def encode_prompt(article: str, max_prompt_tokens: int):
    article_ids = tokenizer(
        str(article),
        add_special_tokens=False,
        truncation=False,
    )["input_ids"]
    available_article_tokens = (
        max_prompt_tokens - len(prefix_ids) - len(generation_suffix_ids)
    )
    if available_article_tokens < 1:
        raise ValueError("The generation budget leaves no room for the article.")

    used_article_ids = article_ids[:available_article_tokens]
    prompt_ids = prefix_ids + used_article_ids + generation_suffix_ids
    return prompt_ids, {
        "article_token_count": len(article_ids),
        "used_article_token_count": len(used_article_ids),
        "article_was_truncated": len(used_article_ids) < len(article_ids),
    }


SECTION_BOUNDARY_PATTERN = re.compile(
    r"^\s*(?:[-*]\s*)?"
    r"(?:Aspects?|Events?|People|Date\s*(?:&|and)\s*Time|"
    r"Locations?|Effects?|Counter\s*measures?|Countermeasures?)\s*:",
    flags=re.IGNORECASE | re.MULTILINE,
)

EXTRA_SECTION_PATTERN = re.compile(
    r"^\s*(?:[-*]\s*)?"
    r"(?:Summary\s+(?:of|and)\s+[^\n:]{1,50}|Analysis|Aspect|"
    r"Athletics|Sports(?:-news)?(?:\s+articles)?)\s*:",
    flags=re.IGNORECASE | re.MULTILINE,
)

def extract_summary_section(generated_output: str) -> str:
    text = str(generated_output).strip()
    text = re.sub(
        r"^(?:\s*Summary\s*:\s*)+",
        "",
        text,
        count=1,
        flags=re.IGNORECASE,
    )
    boundaries = [
        match.start()
        for pattern in (SECTION_BOUNDARY_PATTERN, EXTRA_SECTION_PATTERN)
        if (match := pattern.search(text)) is not None
    ]
    if boundaries:
        text = text[:min(boundaries)]

    sentences = [
        sentence.strip()
        for sentence in re.split(r"(?<=[.!?])\s+|\n+", text.strip())
        if sentence.strip()
    ]
    return " ".join(sentences[:4])


def extract_aspects_section(generated_output: str) -> str:
    text = str(generated_output).strip()
    boundary = SECTION_BOUNDARY_PATTERN.search(text)
    if boundary is None:
        return ""
    return text[boundary.start():].strip()


def generate_article_output(article: str):
    max_input_tokens = MAX_SEQUENCE_LENGTH - MAX_NEW_TOKENS
    prompt_ids, token_info = encode_prompt(
        article,
        max_prompt_tokens=max_input_tokens,
    )
    input_ids = torch.tensor(
        [prompt_ids],
        dtype=torch.long,
        device=model_device,
    )
    attention_mask = torch.ones_like(input_ids)

    start_time = time.perf_counter()
    with torch.inference_mode():
        output_ids = finetuned_model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            num_beams=1,
            repetition_penalty=1.1,
            no_repeat_ngram_size=3,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    elapsed = time.perf_counter() - start_time

    generated_ids = output_ids[0, input_ids.shape[1]:]
    generated_continuation = tokenizer.decode(
        generated_ids,
        skip_special_tokens=True,
    ).strip()
    generated_output = (
        SUMMARY_GENERATION_PREFIX + generated_continuation
    ).strip()
    generated_id_list = generated_ids.tolist()
    ended_with_eos = (
        tokenizer.eos_token_id is not None
        and tokenizer.eos_token_id in generated_id_list
    )
    token_info.update({
        "generated_token_count": len(generated_id_list),
        "generation_hit_token_limit": (
            len(generated_id_list) >= MAX_NEW_TOKENS and not ended_with_eos
        ),
    })
    return generated_output, elapsed, token_info


generated_outputs = []
generation_times = []
generation_token_details = []

for position, article in enumerate(test_df["article"], start=1):
    output, elapsed, token_info = generate_article_output(article)
    generated_outputs.append(output)
    generation_times.append(elapsed)
    generation_token_details.append(token_info)

    if position % 10 == 0 or position == len(test_df):
        print(f"Generated {position}/{len(test_df)} responses")

test_df = test_df.copy()
test_df["generated_text"] = generated_outputs
test_df["generated_summary"] = [
    extract_summary_section(output) for output in generated_outputs
]
test_df["generated_aspects"] = [
    extract_aspects_section(output) for output in generated_outputs
]
test_df["generated_summary_found"] = (
    test_df["generated_summary"].str.strip().ne("")
)
test_df["generated_aspects_found"] = (
    test_df["generated_aspects"].str.strip().ne("")
)

test_df["generation_time_sec"] = generation_times
for detail_key in (
    "article_token_count",
    "used_article_token_count",
    "article_was_truncated",
    "generated_token_count",
    "generation_hit_token_limit",
):
    test_df[detail_key] = [
        details[detail_key] for details in generation_token_details
    ]

test_df[
    ["generated_text", "generated_summary", "generated_aspects"]
].head()

Generated 10/30 responses
Generated 20/30 responses
Generated 30/30 responses


,generated_text,generated_summary,generated_aspects
0,Summary:\nItalian coach Antonio ConTE has been...,Italian coach Antonio ConTE has been criticise...,Aspect:\nEvents:\nPeople : Mario Balorelli; An...
1,Summary:\nChild was injured while trying to he...,Child was injured while trying to help uncle a...,Aspect:\n Event: Colton LeeClegg was assisting...
2,Summary:\nThe Libyan Shield Brigade was fighti...,The Libyan Shield Brigade was fighting against...,Aspect:\nEvents:\nPeople gathered outside the ...
3,Summary:\nDakota Johnson blindfolds her body a...,Dakota Johnson blindfolds her body and archs o...,Aspect:\n Event: Christian Grey met Anastasia ...
4,Summary:\nKaty is a monster.\nShe's a monster ...,Katy is a monster. She's a monster who can't b...,Aspect:\n Event: Katy Perry performs a live co...


## Evaluate summaries and aspects

Summary and combined-aspect quality use ROUGE-1, ROUGE-2, and ROUGE-L. The generated aspect block is evaluated in its original format without parsing individual labels. Readability and compression apply only to the generated summary. These automatic scores do not establish factual correctness.

In [7]:
scorer = rouge_scorer.RougeScorer(
    ["rouge1", "rouge2", "rougeL"],
    use_stemmer=True,
)


def compute_rouge(reference: str, hypothesis: str):
    scores = scorer.score(str(reference), str(hypothesis))
    return {
        "rouge1": scores["rouge1"].fmeasure,
        "rouge2": scores["rouge2"].fmeasure,
        "rougeL": scores["rougeL"].fmeasure,
    }


def format_aspect_value(value) -> str:
    if isinstance(value, list):
        values = [str(item).strip() for item in value if str(item).strip()]
        return "; ".join(values) if values else "Not stated."
    if value is None or not str(value).strip():
        return "Not stated."
    return str(value).strip()


def format_reference_aspects(aspects: dict) -> str:
    lines = ["Aspects:"]
    lines.extend(
        f"{label}: {format_aspect_value(aspects.get(key))}"
        for key, label in ASPECT_LABELS.items()
    )
    return "\n".join(lines)


def word_count(text: str) -> int:
    return max(1, len(str(text).split()))


def has_complete_structure(text: str) -> bool:
    required_headers = [
        "Summary:",
        "Aspects:",
        *(f"{label}:" for label in ASPECT_LABELS.values()),
    ]
    return all(header in str(text) for header in required_headers)


summary_scores = []
aspect_scores = []
flesch_scores = []
compression_ratios = []
structured_complete = []

for row_position, (_, row) in enumerate(test_df.iterrows()):
    summary_scores.append(
        compute_rouge(row["summary"], row["generated_summary"])
    )

    aspect_scores.append(
        compute_rouge(
            format_reference_aspects(row["aspects"]),
            row["generated_aspects"],
        )
    )

    generated_summary = row["generated_summary"]
    flesch_scores.append(
        textstat.flesch_reading_ease(generated_summary)
        if generated_summary
        else 0.0
    )
    compression_ratios.append(
        word_count(generated_summary) / word_count(row["article"])
    )
    structured_complete.append(
        has_complete_structure(generated_outputs[row_position])
    )

test_df["summary_rouge1"] = [score["rouge1"] for score in summary_scores]
test_df["summary_rouge2"] = [score["rouge2"] for score in summary_scores]
test_df["summary_rougeL"] = [score["rougeL"] for score in summary_scores]
test_df["aspect_rouge1"] = [score["rouge1"] for score in aspect_scores]
test_df["aspect_rouge2"] = [score["rouge2"] for score in aspect_scores]
test_df["aspect_rougeL"] = [score["rougeL"] for score in aspect_scores]
test_df["summary_flesch_reading_ease"] = flesch_scores
test_df["summary_compression_ratio"] = compression_ratios
test_df["structured_output_complete"] = structured_complete

## Aggregate and save results

The per-example CSV keeps the complete `generated_text`, the extracted `generated_summary`, the original-format `generated_aspects` block, and diagnostic fields for missing sections and token-limit hits. If the preferred CSV is locked by Excel or another program, the notebook saves a timestamped alternative and reports its path.

In [11]:
metric_series = {
    "Summary ROUGE-1": test_df["summary_rouge1"],
    "Summary ROUGE-2": test_df["summary_rouge2"],
    "Summary ROUGE-L": test_df["summary_rougeL"],
    "Aspects ROUGE-1": test_df["aspect_rouge1"],
    "Aspects ROUGE-2": test_df["aspect_rouge2"],
    "Aspects ROUGE-L": test_df["aspect_rougeL"],
}
metric_series.update({
    "Summary Flesch Reading Ease": test_df["summary_flesch_reading_ease"],
    "Summary Compression Ratio": test_df["summary_compression_ratio"],
    "Generation Time (sec)": test_df["generation_time_sec"],
    "Generated Summary Found Rate": test_df["generated_summary_found"],
    "Generated Aspects Found Rate": test_df["generated_aspects_found"],
    "Structured Output Completion Rate": test_df["structured_output_complete"],
    "Generation Token-Limit Hit Rate": test_df["generation_hit_token_limit"],
    "Article Truncation Rate": test_df["article_was_truncated"],
})

summary_metrics = pd.DataFrame([
    {"metric": metric, "mean": values.mean(), "std": values.std()}
    for metric, values in metric_series.items()
])
print(summary_metrics.to_string(index=False))

result_columns = [
    "article_id",
    "article",
    "summary",
    "article_word_count",
    "aspects",
    "generated_text",
    "generated_summary",
    "generated_aspects",
    "generation_time_sec",
    "generated_token_count",
    "generation_hit_token_limit",
    "article_token_count",
    "used_article_token_count",
    "article_was_truncated",
    "summary_rouge1",
    "summary_rouge2",
    "summary_rougeL",
    "aspect_rouge1",
    "aspect_rouge2",
    "aspect_rougeL",
    "summary_flesch_reading_ease",
    "summary_compression_ratio",
    "generated_summary_found",
    "generated_aspects_found",
    "structured_output_complete",
]


def save_csv_with_fallback(frame: pd.DataFrame, preferred_path: Path) -> Path:
    try:
        frame.to_csv(preferred_path, index=False)
        return preferred_path
    except PermissionError:
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        fallback_path = preferred_path.with_name(
            f"{preferred_path.stem}_{timestamp}{preferred_path.suffix}"
        )
        frame.to_csv(fallback_path, index=False)
        print(
            f"Could not overwrite '{preferred_path}' because it is open or locked. "
            f"Saved to '{fallback_path}' instead."
        )
        return fallback_path


per_example_path = save_csv_with_fallback(
    test_df[result_columns],
    RESULTS_DIR / f"{RESULT_FILE_NAME}.csv",
)
summary_path = save_csv_with_fallback(
    summary_metrics,
    RESULTS_DIR / f"{RESULT_FILE_NAME}_summary.csv",
)

print(f"\nSaved test results to '{per_example_path}'")
print(f"Saved evaluation summary to '{summary_path}'")

                           metric      mean       std
                  Summary ROUGE-1  0.258975  0.087029
                  Summary ROUGE-2  0.056373  0.042987
                  Summary ROUGE-L  0.180582  0.068637
                  Aspects ROUGE-1  0.306052  0.096114
                  Aspects ROUGE-2  0.080202  0.045871
                  Aspects ROUGE-L  0.192581  0.064025
      Summary Flesch Reading Ease 57.990877 13.040516
        Summary Compression Ratio  0.121298  0.080937
            Generation Time (sec)  4.791102  2.532571
     Generated Summary Found Rate  1.000000  0.000000
     Generated Aspects Found Rate  0.966667  0.182574
Structured Output Completion Rate  0.000000  0.000000
  Generation Token-Limit Hit Rate  0.200000  0.406838
          Article Truncation Rate  0.000000  0.000000

Saved test results to 'results\opt125m_cnn_test_results.csv'
Saved evaluation summary to 'results\opt125m_cnn_test_results_summary.csv'
